# Point Source to Far Field Reflector — Benchmark

**Kernel → Restart & Run All** runs the full pipeline end-to-end:

| Step | What happens |
|------|-------------|
| 1 | **Config** — sizes, benchmark case, patch bounds |
| 2 | **Dependencies** — ensure numpy + matplotlib are installed |
| 3 | **Generate** quasi-random point clouds on the sphere |
| 4 | **Compile** C++ via `make` |
| 5 | **Run** the benchmark |
| 6 | **Find output** + define data-loading helpers |
| 7 | **Visualize** (matplotlib, saved as PNG) |
| 8 | **Interactive 3D** ray diagram (Plotly) |

## Step 1 — Configuration

In [ ]:
import os, sys

# ── Point-cloud sizes ─────────────────────────────────────────────────────────
# NK complexity is O(NK²):  1600 → fast (seconds)  |  16488 → full (minutes)
NK       = 1600
NK_small = 200    # warm-start sampler (must be < NK)

# ── Benchmark test-case ───────────────────────────────────────────────────────
# Choose ONE of:
#   'test_3D_SquareToCircle_logcost_MonteCarlo.h'
#   'test_3D_SquareToTwoGaussSide_logcost_MonteCarlo.h'
BENCHMARK = 'test_3D_SquareToCircle_logcost_MonteCarlo.h'

# ── Patch bounds (multiples of π) ─────────────────────────────────────────────
# These must match the sphere regions used in generate_pointclouds.py.
# Defaults: source = north cap (0–60°), target = south cap (120–180°).
SRC_THETA_MIN = 0.0
SRC_THETA_MAX = 1/3   # 60°
SRC_PHI_MIN   = 0.0
SRC_PHI_MAX   = 2.0   # 360°

TGT_THETA_MIN = 2/3   # 120°
TGT_THETA_MAX = 1.0   # 180°
TGT_PHI_MIN   = 0.0
TGT_PHI_MAX   = 2.0

# ── Paths ─────────────────────────────────────────────────────────────────────
REPO_ROOT = os.path.abspath('')
CODE_DIR  = os.path.join(REPO_ROOT, 'BenchmarkCode')

print(f'Python    : {sys.executable}')
print(f'Repo root : {REPO_ROOT}')
print(f'Code dir  : {CODE_DIR}')
print(f'NK={NK}, NK_small={NK_small}')
print(f'Benchmark : {BENCHMARK}')
print(f'Source patch  θ: [{SRC_THETA_MIN}π, {SRC_THETA_MAX}π]  φ: [{SRC_PHI_MIN}π, {SRC_PHI_MAX}π]')
print(f'Target patch  θ: [{TGT_THETA_MIN}π, {TGT_THETA_MAX}π]  φ: [{TGT_PHI_MIN}π, {TGT_PHI_MAX}π]')

## Step 2 — Install dependencies

Installs numpy and matplotlib into **this kernel** via `sys.executable` so the correct
site-packages directory is targeted regardless of the environment.

In [ ]:
import subprocess

def pip(*args):
    cmd = [sys.executable, '-m', 'pip', *args]
    r = subprocess.run(cmd, capture_output=True, text=True)
    for line in (r.stdout + r.stderr).splitlines():
        if any(k in line for k in ('ERROR', 'error', 'WARNING', 'Successfully', 'already')):
            print(line)
    return r.returncode

print('Installing numpy ...')
pip('install', 'numpy>=1.21,<2.0', '--upgrade', '--quiet')

print('Installing matplotlib (force-reinstall to ensure binary compatibility) ...')
pip('install', 'matplotlib', '--upgrade', '--force-reinstall', '--quiet')

r = subprocess.run(
    [sys.executable, '-c',
     'import numpy, matplotlib; print("numpy", numpy.__version__, "| matplotlib", matplotlib.__version__)'],
    capture_output=True, text=True
)
print(r.stdout.strip() or r.stderr.strip())
print('✓ Dependencies ready')

## Step 3 — Generate point clouds

Calls `BenchmarkCode/generate_pointclouds.py` which writes three header files:

| File | Contents |
|------|----------|
| `QuasiMonteCarlo/MonteCarlo_Pointcloud_3D_128.h` | NK source + target points (Halton sequence, uniform in solid angle) |
| `SmallGrid/3D_MonteCarlo_Pointcloud_small.h` | NK_small warm-start points |
| `PushForward/PushForward_Cloud_128.h` | NK push-forward source points |

The script defaults to: **source** = north cap (0–60°), **target** = south cap (120–180°).
Edit `generate_pointclouds.py` to change patch geometry and re-run this cell.

In [ ]:
gen_script = os.path.join(CODE_DIR, 'generate_pointclouds.py')

print(f'Generating point clouds: NK={NK}, NK_small={NK_small} ...')
r = subprocess.run(
    [sys.executable, gen_script, str(NK), str(NK_small)],
    capture_output=True, text=True
)
print(r.stdout.strip())
if r.returncode != 0:
    print('STDERR:', r.stderr)
    raise RuntimeError(f'generate_pointclouds.py failed (exit {r.returncode})')
print('✓ Point clouds written')

## Step 4 — Compile

In [ ]:
import re

# Patch the #include in main.cpp to the chosen benchmark
main_cpp = os.path.join(CODE_DIR, 'main.cpp')
with open(main_cpp) as fh:
    src = fh.read()
patched = re.sub(
    r'#include\s+"Benchmarks/[^"]+"',
    f'#include "Benchmarks/{BENCHMARK}"',
    src, count=1
)
if patched != src:
    with open(main_cpp, 'w') as fh:
        fh.write(patched)
    print(f'Patched main.cpp → {BENCHMARK}')
else:
    print(f'main.cpp already set to {BENCHMARK}')

print('\nCompiling ...')
r = subprocess.run(['make', '-C', CODE_DIR], capture_output=True, text=True)
print(r.stdout)
if r.returncode != 0:
    print('STDERR:', r.stderr)
    raise RuntimeError(f'Compilation failed (exit {r.returncode})')
print('✓ Compiled successfully')

## Step 5 — Run benchmark

The patch-bound arguments are passed as multiples of π (converted to radians inside `main()`).

In [ ]:
import time

print(f'Running benchmark (NK={NK}) ...')
t0 = time.time()

r = subprocess.run(
    [
        os.path.join(CODE_DIR, 'main'),
        str(SRC_THETA_MIN), str(SRC_THETA_MAX),
        str(SRC_PHI_MIN),   str(SRC_PHI_MAX),
        str(TGT_THETA_MIN), str(TGT_THETA_MAX),
        str(TGT_PHI_MIN),   str(TGT_PHI_MAX),
    ],
    cwd=CODE_DIR, capture_output=True, text=True
)
elapsed = time.time() - t0

out = r.stdout
print(out[-4000:] if len(out) > 4000 else out)
if r.returncode != 0:
    print('STDERR:', r.stderr[-2000:])
    raise RuntimeError(f'Benchmark failed (exit {r.returncode})')

print(f'✓ Completed in {elapsed:.1f}s')

## Step 6 — Find output & load helpers

In [ ]:
import glob
import numpy as np

# ── Locate the most-recently written output directory ─────────────────────────
def find_output(code_dir):
    patterns = [
        os.path.join(code_dir, 'Results_*', '**', 'Output_*'),
        os.path.join(code_dir, 'Results_*', '*'),
        os.path.join(code_dir, 'Output_*'),
    ]
    dirs = []
    for p in patterns:
        dirs += [d for d in glob.glob(p, recursive=True) if os.path.isdir(d)]
    return max(dirs, key=os.path.getmtime) if dirs else None

output_dir = find_output(CODE_DIR)
if output_dir is None:
    raise RuntimeError('No output directory found — did Step 5 succeed?')

print(f'Output directory : {output_dir}')
txts = sorted(f for f in os.listdir(output_dir) if f.endswith('.txt'))
print(f'Output files ({len(txts)}): {txts}')

# ── Data-loading helpers (used by Steps 7 & 8) ────────────────────────────────
_j = lambda name: os.path.join(output_dir, name)

def _load_vec(path):
    """Load a .txt file whose first non-empty line is a count header.
    Returns an (N, D) float array, or None if the file is missing."""
    if not os.path.exists(path):
        return None
    rows = []
    with open(path) as fh:
        fh.readline()          # skip header
        for line in fh:
            line = line.strip()
            if line:
                vals = [float(v) for v in line.split()]
                if vals:
                    rows.append(vals)
    return np.array(rows) if rows else None

def _load_pts(path):
    """Load a .txt file with no header — each line is a space-separated row."""
    if not os.path.exists(path):
        return None
    rows = []
    with open(path) as fh:
        for line in fh:
            line = line.strip()
            if line:
                vals = [float(v) for v in line.split()]
                if len(vals) >= 2:
                    rows.append(vals)
    return np.array(rows) if rows else None

print('✓ Helpers defined')

## Step 7 — Visualize (matplotlib)

Runs `visualize.py` in a **fresh subprocess** so any newly installed matplotlib binary is
picked up cleanly. Saved PNGs are then displayed inline.

In [ ]:
from IPython.display import Image, display

print('Running visualization subprocess ...')
r = subprocess.run(
    [
        sys.executable, os.path.join(REPO_ROOT, 'visualize.py'),
        '--output-dir',    output_dir,
        '--benchmark',     BENCHMARK,
        '--nk',            str(NK),
        '--src-theta-min', str(SRC_THETA_MIN),
        '--src-theta-max', str(SRC_THETA_MAX),
        '--src-phi-min',   str(SRC_PHI_MIN),
        '--src-phi-max',   str(SRC_PHI_MAX),
        '--tgt-theta-min', str(TGT_THETA_MIN),
        '--tgt-theta-max', str(TGT_THETA_MAX),
        '--tgt-phi-min',   str(TGT_PHI_MIN),
        '--tgt-phi-max',   str(TGT_PHI_MAX),
    ],
    capture_output=True, text=True
)
print(r.stdout.strip())
if r.returncode != 0:
    print('STDERR:', r.stderr[-3000:])
    raise RuntimeError('Visualization subprocess failed')

for png in ['visualization.png', 'reflector_analysis.png', 'density_comparison.png']:
    path = _j(png)
    if os.path.exists(path):
        print(f'\n{png}')
        display(Image(path))

## Step 8 — Interactive 3D ray diagram (Plotly)

Shows:
- **Gold** — incoming rays from origin O to reflector surface
- **Viridis** — reflector surface (coloured by radius)
- **Blue** — outgoing reflected rays (direction = OT-assigned target)

Drag to rotate, scroll to zoom, click legend items to toggle.

In [ ]:
try:
    import plotly.graph_objects as go
except ImportError:
    import subprocess as _sp, sys as _sys
    _sp.run([_sys.executable, '-m', 'pip', 'install', 'plotly', '--quiet'])
    import plotly.graph_objects as go

# ── Parameters ────────────────────────────────────────────────────────────────
# COST_K must match Generic_3D_logcost_MonteCarlo.h (default 0.7 for reflection)
COST_K  = 0.7
N_RAYS  = 150
T_OUT   = 2.0

# ── Load data ────────────────────────────────────────────────────────────────
x_src = _load_vec(_j('x_MY.txt'))    # (NK, 3) source directions
y_tgt = _load_vec(_j('y_MY.txt'))    # (NK, 3) target directions
ref   = _load_vec(_j('Ref_MY.txt'))  # (NK, 3) reflector surface points
p_src = _load_vec(_j('p_MY.txt'))    # (NK,)   source density weights
g_pot = _load_vec(_j('g_MY.txt'))    # (NK,)   Sinkhorn dual potential g

for name, arr in [('x_MY', x_src), ('y_MY', y_tgt),
                  ('Ref_MY', ref), ('p_MY', p_src), ('g_MY', g_pot)]:
    if arr is None:
        raise RuntimeError(f'{name}.txt not found — re-run Steps 5–6 first.')

p_flat = p_src.flatten()
g_flat = g_pot.flatten()

# ── OT assignment: j*(i) = argmin_j [ c(x_i, y_j) − g[j] ] ─────────────────
dot    = x_src @ y_tgt.T                                     # (NK, NK)
cost   = -np.log(np.clip(1.0 - COST_K * dot, 1e-12, None))  # reflector cost
assign = (cost - g_flat[np.newaxis, :]).argmin(axis=1)       # (NK,)
T_x    = y_tgt[assign]                                       # assigned directions

# ── Subsample active rays ────────────────────────────────────────────────────
active  = np.where(p_flat > 0)[0]
rng     = np.random.default_rng(0)
ray_idx = rng.choice(active, size=min(N_RAYS, len(active)), replace=False)

def _segs(starts, ends):
    xs, ys, zs = [], [], []
    for s, e in zip(starts, ends):
        xs += [s[0], e[0], None]
        ys += [s[1], e[1], None]
        zs += [s[2], e[2], None]
    return xs, ys, zs

# incoming: O → Ref[i]
xi, yi, zi = _segs(np.zeros((len(ray_idx), 3)), ref[ray_idx])
# outgoing: Ref[i] → Ref[i] + T_OUT * direction
xo, yo, zo = _segs(ref[ray_idx], ref[ray_idx] + T_OUT * T_x[ray_idx])

# ── Figure ───────────────────────────────────────────────────────────────────
fig = go.Figure()

# Reflector surface
r_data = _load_vec(_j('R_MY.txt'))
r_int  = r_data.flatten() if r_data is not None else ref[:, 2]
idx    = rng.choice(len(ref), min(len(ref), 4000), replace=False)
fig.add_trace(go.Scatter3d(
    x=ref[idx, 0], y=ref[idx, 1], z=ref[idx, 2], mode='markers',
    marker=dict(size=2, color=r_int[idx], colorscale='Viridis', opacity=0.4),
    name='Reflector surface', hoverinfo='skip',
))

# Incoming rays
fig.add_trace(go.Scatter3d(
    x=xi, y=yi, z=zi, mode='lines',
    line=dict(color='rgba(255,160,20,0.7)', width=2),
    name=f'Incoming rays ({len(ray_idx)})', hoverinfo='skip',
))

# Outgoing rays
fig.add_trace(go.Scatter3d(
    x=xo, y=yo, z=zo, mode='lines',
    line=dict(color='rgba(30,120,255,0.8)', width=2),
    name=f'Reflected rays (t={T_OUT})', hoverinfo='skip',
))

# Point source
fig.add_trace(go.Scatter3d(
    x=[0], y=[0], z=[0], mode='markers',
    marker=dict(size=6, color='red'),
    name='Point source O',
))

fig.update_layout(
    title=dict(
        text=(
            f'<b>Reflected Ray Paths — {BENCHMARK.replace(".h", "")}</b><br>'
            f'<sup>NK={NK}  ·  {len(ray_idx)} rays  ·  COST_K={COST_K}  ·  t={T_OUT}</sup>'
        ),
        x=0.5, xanchor='center',
    ),
    scene=dict(
        xaxis=dict(title='X'), yaxis=dict(title='Y'), zaxis=dict(title='Z'),
        aspectmode='data',
        camera=dict(eye=dict(x=1.4, y=1.4, z=0.8)),
    ),
    legend=dict(itemsizing='constant', x=0, y=1),
    margin=dict(l=0, r=40, t=100, b=0),
    width=960, height=720,
)

fig.show()
print(f'Ray plot ready.  {len(ray_idx)} rays, outgoing length t={T_OUT}.')